## Import

In [ ]:
import os
import sep
import cv2
import glob
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from datetime import datetime
import matplotlib.dates as mdates
import matplotlib.cm as cm
import astroalign as aa
from pandas import read_csv
from ultralytics import YOLO
from pathlib import Path
import shutil
import random

In [ ]:
def load_tj_img(path, denoise=True):
    img = cv2.imread(path, -1).astype(np.float32)
    if not denoise:
        return img
    med = cv2.medianBlur(img, 3)
    sub = img - med
    mask = (sub > med) & (med < med.mean()+med.std())
    oup = img.copy()
    oup[mask] = med[mask]
    return oup

## 结构简图

In [ ]:
img = cv2.imread('dataset/test/xt/still_001.png', -1)
fig, ax = plt.subplots(1,1,figsize=(8,8))
ax.imshow(img[..., 1], cmap='jet')
ax.axis('off')
plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
plt.show()

### 时序变化图

In [ ]:
a = 2048
path_dir = '/mnt/e/Imgs/03-TJStars/150ms-2k2k/decode/'
imgs = sorted(glob.glob(path_dir + '/*.tif'))
aligns = []
for path in tqdm(imgs, desc='Generate Mask'):
    img2 = load_tj_img(path)
    aligns.append(img2)
aligns = np.array(aligns)

In [ ]:
oup = np.max(aligns, axis=0)
vmin, vmax = np.percentile(oup, (0.5,99.5))

plt.figure(figsize=(10,10))
plt.imshow(oup, vmin=vmin, vmax=vmax)
plt.plot(1755, 475, 'ro', ms=1)
plt.plot(1477, 1061, 'ro', ms=1)
plt.plot(250, 125, 'ro', ms=1)
plt.grid()
# plt.axis('off')

In [ ]:
global_bkg = np.median(aligns)
print(global_bkg)
fig, axes = plt.subplots(3, 1, figsize=(12, 6), sharex=True)
axes[0].plot(aligns[:, 475, 1755], label='star')
axes[1].plot(aligns[:, 1061, 1477], label='tar')
axes[2].plot(aligns[:, 1024, 1024], label='bkg')
for ax in axes: ax.legend()

tj1

In [ ]:
path_dir = '/mnt/e/Imgs/03-TJStars/TJ1/decode/'
imgs = glob.glob(path_dir + '/*.tif')
img1 = load_tj_img(imgs[0])
aligns = [img1]
for path in tqdm(imgs[1:], desc='Generate Mask'):
    img2 = load_tj_img(path)
    img21, footprint = aa.register(img2, img1)
    aligns.append(img21)
aligns = np.array(aligns)

In [ ]:
global_bkg = np.median(aligns, axis=(1,2))
print(global_bkg)

In [ ]:
# here: academic-style line plot
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 9,
    'axes.labelsize': 10,
    'axes.titlesize': 10,
    'legend.fontsize': 8,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'figure.dpi': 150,
    'savefig.dpi': 300,
})

fig, axes = plt.subplots(3, 1, figsize=(6.5, 3), sharex=True, constrained_layout=True)
series = [
    ('Star', aligns[:, 3620, 1345], '#1f77b4', '-'),
    ('Target', aligns[:, 1500, 2800], '#d62728', '-'),
    ('Background', aligns[:, 2000, 2100], '#2ca02c', '-'),
]

for ax, (name, y, color, ls) in zip(axes, series):
    x = np.arange(len(y))
    ax.plot(x, y, color=color, lw=1.4, ls=ls, marker='o', ms=2.2, markevery=max(1, len(y)//18), label=name)
    ax.grid(True, which='major', ls='--', lw=0.6, alpha=0.35)
    # ax.spines['center'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_ylabel('Intensity')
    ax.legend(loc='upper right', frameon=True, framealpha=0.9, edgecolor='0.8')

xg = np.arange(len(global_bkg))
axes[2].plot(xg, global_bkg, label='global median', ls='--', lw=1.2, color='0.35')
axes[2].legend(loc='upper right', frameon=True, framealpha=0.9, edgecolor='0.8')
axes[-1].set_xlabel('Frame Index')
# fig.suptitle('Temporal Intensity Profiles at Representative Pixels', y=1.02)
plt.savefig('paper/intensity_vs_frames.pdf', dpi=500)

In [ ]:
oup = np.max(aligns, axis=0)
vmin, vmax = np.percentile(oup, (1,99))

plt.figure(figsize=(10,10))
plt.imshow(oup, vmin=vmin, vmax=vmax)
plt.plot(1345, 3620, 'ro', ms=1)
plt.plot(2800, 1500, 'ro', ms=1)
plt.plot(2000, 2100, 'ro', ms=1)
plt.grid()
# plt.axis('off')